In [169]:
def drawProgressBar(percent, barLen = 20,print_100=False):
    if percent!=1 or print_100 :
        sys.stdout.write("\r")
        #sys.stdout.write('\x1b[2K')
        progress = ""
        for i in range(barLen):
            if i < int(barLen * percent):
                progress += "="
            else:
                progress += " "
        sys.stdout.write("[%s] "%progress + " "*(percent * 100<100) + " "*(percent * 100<10) + "%f%%" %(percent * 100))
        sys.stdout.flush()

In [1]:
from circkit.bitwise import BitwiseCircuit
from circkit.arithmetic import ArithmeticCircuit
from circkit import Circuit, Operation, Param

In [33]:
C = BitwiseCircuit(name="YO",word_size=8)
a = C.add_input("a")
b = C.add_input("b")

x0 = a + b
x1 = a - b + 2
x2 = a * b - 3
x3 = a / b * 5
x4 = a % 3 + b

x5 = x0 << 2
print(x5)
print(x5.operation.shift)

x6 = x1 >> 4
x7 = x2.rol(5)
x8 = x3.ror(6)
x9 = ~x4

x10 = x5 ^^ x6
x11 = x7 | x8
x12 = x9 & x10
x13 = -x11

C.add_output(x12)
C.add_output(x13)

inp = [100, 200]
out = C.evaluate(inp)
print("Circuit output:")
print(out)

C.digraph().view()

<BitwiseCircuit:SHL[shift=2]#15 (2)>
2
Circuit output:
[48, 93]


'Digraph.gv.pdf'

In [34]:
C.name

'YO'

In [171]:
def fun_to_circuit_BitWise(fun,nb_inputs,word_size,is_input_a_list = False):
    C = BitwiseCircuit(word_size=word_size)
    inp_nodes = C.add_inputs(n=nb_inputs, format="x_%d")
    if(is_input_a_list):
        C.add_output(fun(inp_nodes))
    else :
        C.add_output(fun(*inp_nodes))
    return C

In [203]:
def f(a,b,c,d):
    x = a+b
    y = c-d
    z = x*y
    return z

def g(a,b):
    return a+b

In [204]:
def EvenBits(word_size):
    Table = [0]*2^word_size
    power = [1]*word_size
    
    for i in range(1,word_size):
        power[i] = 4*power[i-1]
    
    for i in range(2^word_size):
        b = Integer(i).digits(2)
        Table[i] = sum([power[j]*b[j] for j in range(len(b))])
    return Table

In [174]:
def dict_of_list_to_list(dictionnary):
    L = []
    for x in dictionnary.values():
        L += x
    return L

In [175]:
def create_range_proof(C_Arith,node,cpt_w,dict_decomp):
    
    supposed_node,cpt_w = get_gate_to_LK(C_Arith,cpt_w,dict_decomp)
    
    verif_nodes_are_equal = supposed_node - node
    
    return [verif_nodes_are_equal],supposed_node,cpt_w

In [201]:
def inp_BitW_to_Arith(C_Arith,cpt_x,cpt_w,dict_decomp):
    
    inp = C_Arith.add_input("x_%d"%cpt_x)
    cpt_x += 1
    
    potential_inp,cpt_w = get_gate_to_LK(C_Arith,cpt_w,dict_decomp)
    
    verif_nodes_are_equal = potential_inp - inp
    
    return [verif_nodes_are_equal],potential_inp,cpt_x,cpt_w

In [177]:
def get_inputs(node,var_calc):
    
    a = node.incoming[0]
        
    #a_id_BitW is the index of the left input gate of the sum gate in C_BitW
    a_id_BitW = a.id

    #a is the variable in C_Arith linked to the gate which is the left input of the sum
    a = var_calc[a_id_BitW]

    b = _
    if len(node.incoming) >1 :
        b = node.incoming[1]
        #b_id_BitW is the index of the right input gate of the sum gate in C_BitW
        b_id_BitW = b.id

        #b is the variable in C_Arith linked to the gate which is the right input of the sum
        b = var_calc[b_id_BitW]

    return a,b
            

In [178]:
def get_gate_to_LK(C_Arith,cpt_w,dict_decomp):
    
    c_e = C_Arith.add_input("w_%d"%cpt_w)
    cpt_w += 1
    
    c_o = C_Arith.add_input("w_%d"%cpt_w)
    cpt_w += 1
    
    c = 2*c_o + c_e
    
    dict_decomp[c] = [c_e,c_o]
    return c,cpt_w

In [179]:
def AND_BitW_to_Arith(C_Arith,a,b,c,cpt_w,dict_decomp):
    
    a_e = dict_decomp[a][0]
    a_o = dict_decomp[a][1]
    b_e = dict_decomp[b][0]
    b_o = dict_decomp[b][1]
    
    e_o = dict_decomp[c][0]
    o_o = dict_decomp[c][1]
    
    e_e = C_Arith.add_input("w_%d"%cpt_w)
    cpt_w += 1
    
    dict_decomp["other"] += [e_e]
    
    e = a_e + b_e
    
    is_e_right_evaluated = e - (2*e_o + e_e)
    
    
    
    o_e = C_Arith.add_input("w_%d"%cpt_w)
    cpt_w += 1
    
    dict_decomp["other"] += [o_e]
    
    o = a_o + b_o
    
    is_o_right_evaluated = o - (2*o_o + o_e)
    
    
    return [is_e_right_evaluated,is_o_right_evaluated],cpt_w

In [180]:
def XOR_BitW_to_Arith(C_Arith,a,b,c,cpt_w,dict_decomp):
    
    a_e = dict_decomp[a][0]
    a_o = dict_decomp[a][1]
    b_e = dict_decomp[b][0]
    b_o = dict_decomp[b][1]
    
    e_e = dict_decomp[c][0]
    o_e = dict_decomp[c][1]
    
    e_o = C_Arith.add_input("w_%d"%cpt_w)
    cpt_w += 1
    
    dict_decomp["other"] += [e_o]
    
    e = a_e + b_e
    
    is_e_right_evaluated = e - (2*e_o + e_e)
    
    
    
    o_o = C_Arith.add_input("w_%d"%cpt_w)
    cpt_w += 1
    
    dict_decomp["other"] += [o_o]
    
    o = a_o + b_o
    
    is_o_right_evaluated = o - (2*o_o + o_e)
    
    
    return [is_e_right_evaluated,is_o_right_evaluated],cpt_w

In [181]:
def ADD_BitW_to_Arith(C_Arith,a,b,c,cpt_w,word_size):
    
    add = a + b
    
    k = C_Arith.add_input("w_%d"%cpt_w)
    cpt_w += 1
    
    check_k_in_0_1 =  k * (k-1)
    
    is_right_evaluated = add - (c + k*2^word_size)
    
    return [is_right_evaluated,check_k_in_0_1],cpt_w

In [182]:
def SUB_BitW_to_Arith(C_Arith,a,b,c,cpt_w,word_size):
    return ADD_BitW_to_Arith(C_Arith,b,c,a,cpt_w,word_size)

In [183]:
def MUL_BitW_to_Arith(C_Arith,a,b,c,cpt_w,word_size,dict_decomp):
    
    mul = a * b
    
    k,cpt_w = get_gate_to_LK(C_Arith,cpt_w,dict_decomp)
    
    is_right_evaluated = mul - (c + k*2^word_size)
    
    return [is_right_evaluated],cpt_w

In [187]:
def fun_to_circuit_Arithmetic(fun,nb_inputs,word_size=64,is_input_a_list = False):
    if(word_size%2)!=0:
        print("word_size must be even")
        return 
    
    class ArithmeticCircuitExtended(ArithmeticCircuit):
        class Operations(ArithmeticCircuit.Operations):
            class LK(Operation.Variadic):
                T : Param.Tuple() = ()
                def eval(self,*operands):
                    for v in operands:
                        if not v in self.T:
                            return 1
                    return 0
            
            class AND(Operation.Variadic):
                def eval(self,*operands):
                    for b in operands:
                        if b != 0:
                            return False
                    return True
            
            
    C_BitW = fun_to_circuit_BitWise(fun,nb_inputs,word_size,is_input_a_list = False)
    
    C_BitW.digraph().view()
    p = 52435875175126190479447740508185965837690552500527637822603658699938581184513
    
    C_Arith = ArithmeticCircuitExtended(base_ring=GF(p))
    
    T = EvenBits(word_size//2)
    
    #cpt_x is the number of public inputs we already have met
    cpt_x = 0
    
    #cpt_w is the number of private inputs we already have met
    cpt_w = 0
    
    #var_calc is the list of the variables obtained by an add_input operations or a arithmetic computation
    var_calc = []
    
    #var_verif is the list of the variables obtained by a RangeProof operation
    var_verif = []
    
    #to_LK_at_the_end is the list of the variables on which we must do a lookup (at the end)
    to_LK_at_the_end = []
    
    #dict_decomp is a dictionnary containing at x the value x_e and x_o
    dict_decomp = {}
    
    #Initialisation for later
    dict_decomp["other"] = []
    
    for node in C_BitW.nodes :
        
        if node.is_INPUT():
        
            to_verif,c,cpt_x,cpt_w = inp_BitW_to_Arith(C_Arith,cpt_x,cpt_w,dict_decomp)
            
            var_calc.append(c)  
            var_verif += to_verif
            
        elif node.is_CONST():
            
            c = C_Arith.add_const(node.operation.value.integer_representation())
            var_calc.append(c)
            
        else : 

            a,b = get_inputs(node,var_calc)
            c,cpt_w = get_gate_to_LK(C_Arith,cpt_w,dict_decomp)
            var_calc.append(c)

            if node.is_AND():

                """
                if a not in dict_decomp:
                    to_verif,a,cpt_w = create_range_proof(C_Arith,a,cpt_w,dict_decomp)
                    var_verif += to_verif

                if b not in dict_decomp:
                    to_verif,b,cpt_w = create_range_proof(C_Arith,b,cpt_w,dict_decomp)
                    var_verif += to_verif 
                    """
                
                to_verif,cpt_w = AND_BitW_to_Arith(C_Arith,a,b,c,cpt_w,dict_decomp)

                var_verif += to_verif

            #elif node.is_OR():

            elif node.is_XOR():
                if a not in dict_decomp:
                    to_verif,a,cpt_w = create_range_proof(C_Arith,a,cpt_w,dict_decomp)
                    var_verif += to_verif

                if b not in dict_decomp:
                    to_verif,b,cpt_w = create_range_proof(C_Arith,b,cpt_w,dict_decomp)
                    var_verif += to_verif 

                to_verif,cpt_w = XOR_BitW_to_Arith(C_Arith,a,b,c,cpt_w,dict_decomp)

                var_verif += to_verif

            #elif node.is_NOT():

            #elif node.is_SHL():

            #elif node.is_SHR():

            #elif node.is_ROL():

            #elif node.is_ROR():

            elif node.is_ADD(): 

                to_verif,cpt_w = ADD_BitW_to_Arith(C_Arith,a,b,c,cpt_w,word_size)

                var_verif += to_verif

            elif node.is_SUB():

                to_verif,cpt_w = SUB_BitW_to_Arith(C_Arith,a,b,c,cpt_w,word_size)

                var_verif += to_verif

            elif node.is_MUL():

                to_verif,cpt_w = MUL_BitW_to_Arith(C_Arith,a,b,c,cpt_w,word_size,dict_decomp)

                var_verif += to_verif


            #elif node.is_DIV():

            #elif node.is_MOD():

            #elif node.is_NEG():

        if node.is_OUTPUT():

            C_Arith.add_output(c)
            

    list_of_nodes_to_LK = dict_of_list_to_list(dict_decomp)
    big_LK = C_Arith.LK(T)(*list_of_nodes_to_LK)
    
    is_all_and = C_Arith.AND()(*(var_verif+[big_LK]))
    C_Arith.add_output(is_all_and)
    return C_Arith

In [185]:
def decomp(x,word_size):
    X = Integer(x).digits(2)
    X += [0]*(word_size-len(X))
    
    X_e = [X[even] for even in range(0,word_size,2)]
    X_o = [X[odd] for odd in range(1,word_size,2)]
    
    x_e = sum([4^i*X_e[i] for i in range(word_size//2)])
    x_o = sum([4^i*X_o[i] for i in range(word_size//2)])
    
    return x_e,x_o

In [197]:
def witness(Trace_BitW,word_size=64):
    
    w = []
    
    for node in Trace_BitW:
        
        if node.is_INPUT():
            
            inp = T[node]
            
            inp_e,inp_o = decomp(inp,word_size)
            
            w.append(inp_e)
            w.append(inp_o)
            
        #if node.is_CONST():
            
        elif node.is_AND():
            a = Trace_BitW[node.incoming[0]]
            b = Trace_BitW[node.incoming[1]]
            
            a_e,a_o = decomp(a,word_size)
            b_e,b_o = decomp(b,word_size)
            o_e,o_o = decomp(a_o+b_o,word_size)
            e_e,e_o = decomp(a_e+b_e,word_size)
            
            w.append(e_o)
            w.append(o_o)
            w.append(a_e)
            w.append(a_o)
            w.append(b_e)
            w.append(b_o)
            w.append(o_e)
            w.append(e_e)
            
        #elif node.is_OR():
            
        elif node.is_XOR():
            a = Trace_BitW[node.incoming[0]]
            b = Trace_BitW[node.incoming[1]]
            
            a_e,a_o = decomp(a,word_size)
            b_e,b_o = decomp(b,word_size)
            o_e,o_o = decomp(a_o+b_o,word_size)
            e_e,e_o = decomp(a_e+b_e,word_size)
            
            w.append(o_e)
            w.append(e_e)
            w.append(a_e)
            w.append(a_o)
            w.append(b_e)
            w.append(b_o)
            w.append(e_o)
            w.append(o_o)
            
        #elif node.is_NOT():
            
        #elif node.is_SHL():
            
        #elif node.is_SHR():
        
        #elif node.is_ROL():
        
        #elif node.is_ROR():
            
        elif node.is_ADD():
            
            a = Trace_BitW[node.incoming[0]]
            b = Trace_BitW[node.incoming[1]]
            
            c = Integer(a+b)
            
            c_e,c_o = decomp(c,word_size)
            
            if(c>=2^word_size):
                k = 1
            else :
                k = 0
                
            w.append(c_e)
            w.append(c_o)
            w.append(k)
            
        elif node.is_SUB():
            a = Trace_BitW[node.incoming[0]]
            b = Trace_BitW[node.incoming[1]]
            
            c = Integer(a-b)%2^word_size
            
            c_e,c_o = decomp(c,word_size)
            
            if(b+c>=2^word_size):
                k = 1
            else :
                k = 0
                
            w.append(c_e)
            w.append(c_o)
            w.append(k)
            
        elif node.is_MUL():
            a = Trace_BitW[node.incoming[0]]
            b = Trace_BitW[node.incoming[1]]
            
            c = Integer(a*b)
            
            c_e,c_o = decomp(c,word_size)
            
            k = c //2^word_size
            
            k_e,k_o = decomp(k,word_size)
                
            w.append(c_e)
            w.append(c_o)
            w.append(k_e)
            w.append(k_o)
            
        #elif node.is_DIV():
            
        #elif node.is_MOD():
            
        #elif node.is_NEG():
        
    return w

In [206]:
#word_size = 2*randint(1,16);print(word_size)
word_size = 4
C_BitW = fun_to_circuit_BitWise(g,2,word_size)
C_Arith = fun_to_circuit_Arithmetic(g,2,word_size);C_Arith.digraph().view()
#C = fun_to_circuit_Arithmetic(f,4,8);C.digraph().view()

'Digraph.gv.pdf'

In [167]:
T = EvenBits(word_size//2)
total = len(T)^5*2^word_size*2^word_size
i = 0
for inp1 in range(0,2^word_size):
    for inp2 in range(0,2^word_size):
        
        inp = [inp1,inp2]

        y = C_BitW.evaluate(inp)

        T_BitW = C_BitW.trace(inp)
        w = witness(T_BitW,word_size)

        for c_e in T:
            for c_o in T:
                for a_e in T:
                    for a_o in T:
                        for b_e in T:
                            drawProgressBar(float(i)/total)
                            i += 1
                            for b_o in T:
                                for o_e in T:
                                    for e_e in T:
                                        if C_Arith.evaluate(inp+[c_e,c_o,a_e,a_o,b_e,b_o,e_e,o_e])==y+[True]:
                                            if [c_e,c_o,a_e,a_o,b_e,b_o,o_e,e_e]!=w :
                                                print(C_Arith.trace(inp+[c_e,c_o,a_e,a_o,b_e,b_o,o_e,e_e]))
                                                print("inp =",inp)
                                                print("w =",w)
                                                print("pb ",[c_e,c_o,a_e,a_o,b_e,b_o,o_e,e_o])
                                                print("a = 2*a_o + a_e")
                                                print(inp1,"=", 2*a_o + a_e)
                                                print("b = 2*b_o + b_e")
                                                print(inp2,"=", 2*b_o + b_e)
                                                print("a_o + b_o = 2*c_o + o_e")
                                                print(a_o + b_o, "=", 2*c_o + o_e)
                                                print("a_e + b_e = 2*c_e + e_e")
                                                print(a_e + b_e, "=", 2*c_e + e_e)
                                                print("\n")
                                                

[=================== ]  99.999619%

In [65]:
word_size = 4
C_BitW = fun_to_circuit_BitWise(f,4,word_size)
C_Arith = fun_to_circuit_Arithmetic(f,4,word_size);C_Arith.digraph().view()

'Digraph.gv.pdf'

In [66]:
T = EvenBits(word_size//2)
print(T)
for inp1 in range(0,2^word_size):
    for inp2 in range(0,2^word_size):
        for inp3 in range(0,2^word_size):
            for inp4 in range(0,2^word_size):
    
                inp = [inp1,inp2,inp3,inp4]

                y = C_BitW.evaluate(inp)

                T_BitW = C_BitW.trace(inp)
                w = witness(T_BitW,word_size)
        

                for k1 in [0,1]:
                    for c_e1 in T:
                        for c_o1 in T:
                            for k2 in [0,1]:
                                for c_e2 in T:
                                    for c_o2 in T:
                                        for k_e3 in T:
                                            for k_o3 in T:
                                                for c_e3 in T:
                                                    for c_o3 in T:
                                                        if C_Arith.evaluate(inp+[c_e1,c_o1,k1,c_e2,c_o2,k2,c_e3,c_o3,k_e3,k_o3])==y+[True]:
                                                            if [c_e1,c_o1,k1,c_e2,c_o2,k2,c_e3,c_o3,k_e3,k_o3]!=w :
                                                                print()
                                                                print(inp)
                                                                print([c_e1,c_o1,k1,c_e2,c_o2,k2,c_e3,c_o3,k3])
                                                                print(C_Arith.evaluate(inp+[c_e1,c_o1,k1,c_e2,c_o2,k2,c_e3,c_o3,k3]))

[0, 1, 4, 5]


KeyboardInterrupt: 

In [2]:
K = GF(13)
C = ArithmeticCircuit()

a,b,c,d = C.add_inputs(n=4,format="inp_%d")

x1 = a+b
x2 = a+b
x3 = x1 + x2

C.add_output(x3)

In [26]:
C.node_counter

4

In [3]:
C.digraph().view()

'Digraph.gv.pdf'

In [196]:
inp = [0,1,7,3]
T = C.trace(inp)
for node in T :
    print(T[node])

0
1
7
3
3
4
7
3


In [ ]:
C.digraph().view()

In [ ]:
inp = [1,1,2,3]

In [ ]:
C.evaluate(inp)

In [ ]:
for node in C:
    print(node)

In [ ]:
for node in C:
    if(node.is_CONST()):
        print(node.operation.value.integer_representation())

In [ ]:
print(C.inputs)
L = []
for node in C:
    if node.is_INPUT():
        L.append(node)
print("\n")
print(L==C.inputs)
print("\n")
print(L)

In [ ]:
for node in C.inputs:
    print(node.)

In [ ]:
from circkit.arithmetic import ArithmeticCircuit

In [ ]:




#class ArithmeticCircuit(Circuit):

            
        """class EXP(Operation.Unary):
            power : Param.Int(min_value=0) = 5
            def eval(self, a):
                return a**self.power"""
#ArithmeticCircuit.Operations.LK = LK
#Operations.LUT

In [9]:
K = GF(13)
C = ArithmeticCircuitExte(base_ring=K)

a,b,c,d = C.add_inputs(n=4,format="inp_%d")

x1 = a-b
#x2 = c-d

T = [0,1,2,3,4,5,6,7]

T = tuple([K.fetch_int(v) for v in T])

x = C.EXP(4)(c)
y = C.LK()(a)

#y = a.lookup_in(T)
#y = C.LUT(T)(a)
C.add_output([x,y])
#C.add_output([x1,x2])

NameError: name 'ArithmeticCircuitExte' is not defined

In [ ]:
L=67463809.digits(2);L

In [ ]:
sum([2^i*L[i] for i in range(len(L))])

In [ ]:
quit()

In [42]:
dicti = {}

In [43]:
dicti["bla"]=3

In [46]:
"bla" in dicti

True

In [27]:
dicti["bla"]=[1,2]
dicti["blabla"]=[3,4]

In [28]:
dicti

{'bla': [1, 2], 'blabla': [3, 4]}

In [32]:
for x in dicti.values():
    print(x)

[1, 2]
[3, 4]
